# ADS Bibliometric Analysis Pipeline

This notebook is the single entry point for the `ads_bib` package.
It executes all steps sequentially, with configuration cells between steps
so that downstream parameters can be set based on upstream results.

**Pipeline Steps:**
1. Search & Export â€” Query ADS API, resolve bibcodes to metadata
2. Translation â€” Detect languages, translate non-English titles/abstracts
3. Tokenization â€” Create full_text, tokenize with spaCy
4. AND â€” Optional external author name disambiguation
5. Topic Modeling & Curation â€” BERTopic + datamapplot + cluster removal
6. Citation Networks â€” Direct, Co-Citation, Bibliographic Coupling, Author Co-Citation


---
## Setup

In [1]:
import time

from ads_bib.notebook import get_notebook_session

_pipeline_start = time.time()


In [2]:
import logging

root_logger = logging.getLogger()
root_logger.setLevel(logging.INFO)
formatter = logging.Formatter("%(message)s")

if root_logger.handlers:
    for handler in root_logger.handlers:
        handler.setLevel(logging.INFO)
        handler.setFormatter(formatter)
else:
    handler = logging.StreamHandler()
    handler.setLevel(logging.INFO)
    handler.setFormatter(formatter)
    root_logger.addHandler(handler)

logger = logging.getLogger("pipeline")

---
## Global Run Control

Set run-level switches here. Phase-specific parameters are configured in each phase section below.


In [3]:
# -- RUN CONFIGURATION --
# Set RESET_SESSION=True when you want a fresh run directory.
RESET_SESSION = False
RUN = {
    "run_name": "ADS_Curation_Run",
    "random_seed": 42,
    "openrouter_cost_mode": "hybrid",
}

session = get_notebook_session(
    run_name=RUN["run_name"],
    reset=RESET_SESSION,
    start_time=_pipeline_start,
)
session.set_section("run", RUN)


Run initialized: run_20260309_202232_ADS_Curation_Run
All run outputs will be saved to: c:\Users\rapha\Documents\Studium\Promotionsstudium\MPIWG\2_Notebooks\ADS_Pipeline\runs\run_20260309_202232_ADS_Curation_Run
Snapshot of configuration saved to config_used.yaml (9 parameters tracked).
Snapshot of configuration saved to config_used.yaml (9 parameters tracked).


---
# Phase 1: Search & Export

Query the NASA ADS API for publications matching a search string,
then resolve all bibcodes (publications + references) into full metadata.

### 1.1 Search Configuration

Set query, years, and retrieval limits for your research question.
These values define the corpus scope for all downstream steps.


In [4]:
# --- SEARCH CONFIGURATION ---
# Modify the query string for your research question.
# See: https://ui.adsabs.harvard.edu/help/search/search-syntax

Set_A = "docs(library/P0hyiLw0T8qsyHSBpWigJA)"
Set_B = f"citations({Set_A}) AND year:1915-2000"
Set_C = f"citations(citations({Set_A})) AND year:1915-2000"
Set_D = f"({Set_A}) OR ({Set_B}) OR ({Set_C})"
Set_T = "(docs(library/_-RCjJotRKCZC1PP03MqwA) AND year:1915-2000)"

gravity_relativity_terms = [
    '(general AND relativi*)', 'gravit*', '(allgemein* AND relativi*)',
    '"relativité générale"', '"teoria della relatività"',
    '"Gravité quantique"', '"Gravità quantistica"',
    '(einheitlich* AND feld*)', 'Quantengravitation', '"champ unifié"',
    '(unified AND field*)', '"quantum gravity"', 'cosmo*', 'Kosmo*',
]
Set_E = f"abs:({' OR '.join(gravity_relativity_terms)}) AND year:1915-2000"

SEARCH = {
    "query": f"({Set_D}) OR ({Set_T}) OR ({Set_E})",
    "ads_token": None,
    "refresh_search": True,
    "refresh_export": True,
}
# Example quick focus query:
SEARCH["query"] = 'author:"Treder, H*"'
session.set_section("search", SEARCH)


Config changed; invalidated in-memory state from stage 'search'.
Snapshot of configuration saved to config_used.yaml (9 parameters tracked).


Alternative search ideas:

- Quick focus query: `SEARCH["query"] = 'author:"Treder, H*"'`
- Broader historical build: combine `Set_A`, `Set_B`, `Set_C`, `Set_T`, and a wider anchored `Set_E`.
- Update only `SEARCH["query"]`, then rerun the search/export cells.


### 1.2 Execute Search

Run the ADS search and persist bibcodes/results to this run folder.
This creates a reproducible input snapshot for later phases.


In [5]:
session.run_stage("search")


=== Stage: search ===
Starting ADS search ...
  382 records fetched ...
Done. Bibcodes: 382 | Unique refs: 493 | Total refs: 788
Saved: search_results_20260309_202234.pkl
Latest: search_results_latest.pkl


### 1.3 Export & Resolve Metadata

Resolve publications and references to structured metadata tables.
This normalizes schemas before translation/tokenization/topic modeling.


In [6]:
session.run_stage("export")


=== Stage: export ===
=== Exporting publications ===
Exporting 382 bibcodes in 1 chunks (5 workers) ...


Exporting:   0%|          | 0/382 [00:00<?, ?it/s]

Publications: 382 records
=== Exporting references (493 unique) ===
Exporting 493 bibcodes in 1 chunks (5 workers) ...


Exporting:   0%|          | 0/493 [00:00<?, ?it/s]

References: 493 records


In [7]:
if session.publications is not None:
    preview_cols = [
        c for c in ("Bibcode", "Year", "Author", "Title", "References")
        if c in session.publications.columns
    ]
    logger.info(
        "Phase 1 preview: publications=%s rows, columns=%s",
        f"{len(session.publications):,}",
        len(session.publications.columns),
    )
    if preview_cols:
        display(session.publications[preview_cols].head(10))
    else:
        display(session.publications.head(10))


Phase 1 preview: publications=382 rows, columns=18


,Bibcode,Year,Author,Title,References
0,1971grun.book.....T,1971,"[Treder, H.J.]",Gravitationstheorie und Äquivalenzprinzip.,[]
1,1967AnP...475..194T,1967,"[Treder, HansJürgen]",Das makroskopische Gravitationsfeld in der ein...,"[1916AnP...354..769E, 1940PhRv...57..147R, 195..."
2,1983AN....304..145T,1983,"[Treder, H.J.]",The problem of the Grand Unification Theory,[1950sts..book.....S]
3,1957AnP...454..369T,1957,"[Treder, H.]",Stromladungsdefinition und elektrische Kraft i...,[1953PhRv...92.1567C]
4,1972drdt.book.....T,1972,"[Treder, H.J.]",Die Relativität der Trägheit.,[]
5,1997GReGr..29..455V,1997,"[von Borzeszkowski, HorstHeino, Treder, HansJü...",The Weyl-Cartan Space Problem in Purely Affine...,"[1950sts..book.....S, 1971GReGr...2..313T, 199..."
6,1975AnP...487..383T,1975,"[Treder, H.J.]",Zur unitarisierten Gravitationstheorie mit lan...,"[1963ForPh..11...81T, 1973AnP...485..229T, 197..."
7,1981AN....302..275T,1981,"[Treder, H. J., Jackisch, G.]",On Soldners Value of Newtonian Deflection of L...,"[1970AnP...480..315T, 1974AnP...486..325T]"
8,1988mqg..book.....V,1988,"[von Borzeszkowski, H.H., Treder, H.J.]",The meaning of quantum gravity.,[]
9,1980AnP...492..250T,1980,"[Treder, H.J.]",Einstein's Hermitian theory of relativity as u...,"[1957AnP...454..369T, 1975AnP...487..375T]"


---
# Phase 2: Translation

Detect non-English titles and abstracts with fasttext,
then translate them using either OpenRouter (API) or a local HuggingFace model.

### 2.1 Translation Configuration

Choose provider/model and translation target language.
Keep settings explicit so costs and outputs are reproducible.


In [8]:
# -- TRANSLATION CONFIGURATION --
# Providers: openrouter, gguf, nllb
TRANSLATE = {
    "enabled": True,
    "provider": "openrouter",
    "model": "google/gemini-3-flash-preview",
    "api_key": None,
    "max_workers": 10,
    "max_tokens": 2048,
    "fasttext_model": str(session.paths["models"] / "lid.176.bin"),
}
session.set_section("translate", TRANSLATE)


Config changed; invalidated in-memory state from stage 'translate'.
Snapshot of configuration saved to config_used.yaml (9 parameters tracked).


### 2.2 Language Detection + Translation

Detect language tags and translate non-English rows for the current exported dataset.
This stage only works on the current export state or a valid translated snapshot of the same stage.


In [9]:
session.run_stage("translate")


=== Stage: translate ===
fasttext-wheel detected NumPy 2 copy incompatibility; using compatibility path.
  Title: 183 non-English entries detected
  Abstract: 35 non-English entries detected
  Title: 183 non-English entries detected
  Abstract: 36 non-English entries detected
  Title: translating 183 entries with openrouter/google/gemini-3-flash-preview ...


  Title:   0%|          | 0/183 [00:00<?, ?it/s]

  Abstract: translating 35 entries with openrouter/google/gemini-3-flash-preview ...


  Abstract:   0%|          | 0/35 [00:00<?, ?it/s]

  Total translation cost: $0.0300 (218/218 priced; direct=218, fetched=0, mode=hybrid)
  Title: translating 183 entries with openrouter/google/gemini-3-flash-preview ...


  Title:   0%|          | 0/183 [00:00<?, ?it/s]

  Abstract: translating 36 entries with openrouter/google/gemini-3-flash-preview ...


  Abstract:   0%|          | 0/36 [00:00<?, ?it/s]

  Total translation cost: $0.0320 (219/219 priced; direct=219, fetched=0, mode=hybrid)
Translated checkpoint saved to global cache and local run folder.


### 2.3 Preview Translated Fields

Inspect translated text fields after the translate stage has finished.


In [10]:
if session.publications is not None:
    preview_cols = [
        c for c in ("Title", "Title_en", "Abstract", "Abstract_en")
        if c in session.publications.columns
    ]
    if preview_cols:
        display(session.publications[preview_cols].head(5))


,Title,Title_en,Abstract,Abstract_en
0,Gravitationstheorie und Äquivalenzprinzip.,Theory of Gravitation and Principle of Equival...,,
1,Das makroskopische Gravitationsfeld in der ein...,The Macroscopic Gravitational Field in the Uni...,A theory of the gravitation field is proposed ...,A theory of the gravitation field is proposed ...
2,The problem of the Grand Unification Theory,The problem of the Grand Unification Theory,The evolution and fundamental questions of phy...,The evolution and fundamental questions of phy...
3,Stromladungsdefinition und elektrische Kraft i...,Definition of Electric Charge and Electric For...,"Wie Infeld 1950 gezeigt hat, ergibt sich aus d...","As Infeld showed in 1950, the strong system of..."
4,Die Relativität der Trägheit.,The Relativity of Inertia.,,


---
# Phase 3: Tokenization

Create `full_text` (Title + Abstract) and tokenize publications with spaCy.
References are intentionally skipped in this phase for runtime stability.

In [11]:
import os

# -- TOKENIZATION CONFIGURATION --
TOKENIZE = {
    "enabled": True,
    "spacy_model": "en_core_web_md",
    "batch_size": 512,
    "n_process": min(max((os.cpu_count() or 1) - 1, 1), 8),
    "disable": ("ner", "parser", "textcat"),
    "fallback_model": "en_core_web_md",
    "auto_download": True,
}
session.set_section("tokenize", TOKENIZE)


Config changed; invalidated in-memory state from stage 'tokenize'.
Snapshot of configuration saved to config_used.yaml (9 parameters tracked).


In [12]:
session.run_stage("tokenize")


=== Stage: tokenize ===
Tokenizing 382 documents with en_core_web_md (n_process=8, batch_size=512) ...


Tokenization:   0%|          | 0/382 [00:00<?, ?it/s]

  Done.
Tokenized snapshot saved (publications tokenized; refs retained without tokenization).


In [13]:
if session.refs is not None:
    logger.info(
        "References dataset available at %s",
        session.run.paths["data"] / "references.parquet",
    )


References dataset available at c:\Users\rapha\Documents\Studium\Promotionsstudium\MPIWG\2_Notebooks\ADS_Pipeline\runs\run_20260309_202232_ADS_Curation_Run\data\references.parquet


---
# Phase 4: Author Name Disambiguation

Optionally run an external AND package on the curated source datasets.
If disabled, the pipeline stores a Phase-4 checkpoint unchanged and continues.


In [14]:
AUTHOR_DISAMBIGUATION = {
    "enabled": False,
    "model_bundle": None,
    "dataset_id": None,
    "force_refresh": False,
    "infer_stage": "full",
}
session.set_section("author_disambiguation", AUTHOR_DISAMBIGUATION)


Snapshot of configuration saved to config_used.yaml (9 parameters tracked).


In [15]:
session.run_stage("author_disambiguation")


=== Stage: author_disambiguation ===
Disambiguated snapshot saved (publications, references).
Author disambiguation disabled; passthrough snapshot saved.


---
# Phase 5: Topic Modeling & Curation

Use BERTopic to discover thematic clusters, visualize with datamapplot,
then remove unwanted clusters to curate the dataset.

**Important:** Set parameters below based on your dataset size from Phase 1.

### 5.1 Embedding Configuration

Configure embedding provider/model and optional sampling.
These settings strongly affect topic quality and runtime/cost.

| Provider | Model Examples | Cost | Notes |
|----------|----------------|------|-------|
| `local` | `google/embeddinggemma-300m`, `BAAI/bge-large-en-v1.5` | None | CPU/GPU, no API needed |
| `huggingface_api` | `huggingface/BAAI/bge-large-en-v1.5` | Per-token | HF Inference API via LiteLLM |
| `openrouter` | `openai/text-embedding-3-large`, `google/gemini-embedding-001` | Per-token | Central billing, thread-pool concurrency |

**Caching**: Embeddings are cached to `data/cache/embeddings/` with SHA-256 fingerprint validation.
Changing the dataset or model triggers automatic recomputation.


In [16]:
# -- EMBEDDING CONFIGURATION --
# Providers: openrouter, huggingface_api, local
TOPIC_MODEL = {
    "sample_size": None,
    "embedding_provider": "openrouter",
    "embedding_model": "google/gemini-embedding-001",
    "embedding_api_key": None,
    "embedding_batch_size": 64,
    "embedding_max_workers": 20,
}
session.set_section("topic_model", TOPIC_MODEL)


Snapshot of configuration saved to config_used.yaml (9 parameters tracked).


### 5.2 Compute Embeddings

Create or load cached embeddings for `full_text`.
Caching keeps repeated runs fast and reproducible.


In [17]:
logger.info(
    "Topic stages reuse shared snapshots plus the embedding and reduction caches."
)


Topic stages reuse shared snapshots plus the embedding and reduction caches.


In [18]:
session.run_stage("embeddings")


=== Stage: embeddings ===
  Embedding cache mismatch for embeddings_openrouter_google_gemini-embedding-001.npz. Recomputing (cached n_docs=183516, current n_docs=382; cached provider/model=openrouter/google/gemini-embedding-001, current=openrouter/google/gemini-embedding-001).
  Computing embeddings with openrouter/google/gemini-embedding-001 ...


Embedding (OpenRouter):   0%|          | 0/6 [00:00<?, ?it/s]

  Saved: embeddings_openrouter_google_gemini-embedding-001.npz
  Embeddings: (382, 3072)
  embeddings | tokens=31,197 | cost=n/a


### 5.3 Dimensionality Reduction Configuration

Two projections are computed: **5D** (HDBSCAN clustering input) and **2D** (visualization & KDE).

| Method | Strengths | Weaknesses |
|--------|-----------|------------|
| `pacmap` | Fast, good local/global balance | No `densmap` (density-preserving mode) |
| `umap` | Supports `densmap=True` for KDE, better hierarchical structure | Slower, sensitive to `n_neighbors` |

**Key parameters:**

| Parameter | Default | Guidance |
|-----------|---------|----------|
| `n_neighbors` | 80 (PaCMAP) / 80 (UMAP) | Higher = more global structure; lower = more local detail |
| `min_dist` | 0.05 (UMAP only) | Lower = tighter clusters in 2D. Library default 0.1 is too loose for bibliometric data |
| `metric` | `angular`/`cosine` | PaCMAP uses `angular` (auto-converted from `cosine`) |
| `densmap` | `False` (UMAP only) | Set `True` in `PARAMS_2D` if you plan KDE density analysis downstream |

> **Tip**: If you plan KDE analysis on the 2D coordinates, use `method="umap"` with
> `PARAMS_2D = dict(..., densmap=True)`. Without densmap, UMAP inflates dense regions
> in the 2D projection, distorting density estimates.


In [19]:
TOPIC_MODEL = {
    **TOPIC_MODEL,
    "reduction_method": "pacmap",
    "params_5d": {
        "n_neighbors": 80,
        "metric": "angular",
        "random_state": RUN["random_seed"],
    },
    "params_2d": {
        "n_neighbors": 80,
        "metric": "angular",
        "random_state": RUN["random_seed"],
    },
}
session.set_section("topic_model", TOPIC_MODEL)


Config changed; invalidated in-memory state from stage 'reduction'.
Snapshot of configuration saved to config_used.yaml (9 parameters tracked).


In [20]:
session.run_stage("reduction")


=== Stage: reduction ===


Reduction (PACMAP):   0%|          | 0/2 [00:00<?, ?it/s]

  Reduction cache mismatch for reduced_5d_openrouter_google_gemini-embedding-001_pacmap_nn80_minddef_metricangular_rs42.npz. Recomputing.
  Computing 5d_openrouter_google_gemini-embedding-001_pacmap_nn80_minddef_metricangular_rs42 with PACMAP ...
Loading faiss with AVX2 support.
Successfully loaded faiss with AVX2 support.
Note: `n_components != 2` have not been thoroughly tested.
  Saved: reduced_5d_openrouter_google_gemini-embedding-001_pacmap_nn80_minddef_metricangular_rs42.npz
  Reduction cache mismatch for reduced_2d_openrouter_google_gemini-embedding-001_pacmap_nn80_minddef_metricangular_rs42.npz. Recomputing.
  Computing 2d_openrouter_google_gemini-embedding-001_pacmap_nn80_minddef_metricangular_rs42 with PACMAP ...
  Saved: reduced_2d_openrouter_google_gemini-embedding-001_pacmap_nn80_minddef_metricangular_rs42.npz
  5D shape: (382, 5), 2D shape: (382, 2)


### 5.4 Clustering Configuration

HDBSCAN discovers topic clusters based on density in the 5D space.

| Parameter | Default | Guidance |
|-----------|---------|----------|
| `MIN_CLUSTER_SIZE` | `max(15, n_docs * 0.001)` | Controls granularity: ~0.1% of docs. Lower = more (smaller) topics |
| `min_samples` | 2–3 | Lower = fewer outliers but noisier clusters. Higher = stricter density |
| `cluster_selection_method` | `"eom"` | Excess of Mass: selects most persistent clusters |
| `cluster_selection_epsilon` | 0.02–0.05 | Absorbs border points; higher = larger clusters, fewer outliers |

**Backend choice:**
- `fast_hdbscan`: Fastest, but no `prediction_data` or `gen_min_span_tree` (no `approximate_predict()`).
- `hdbscan`: Supports `prediction_data=True` for predicting new documents and `gen_min_span_tree=True` for hierarchy analysis.

`BASE_MIN_CLUSTER_SIZE` is only used by Toponymy for micro-cluster granularity.


In [21]:
TOPIC_MODEL = {
    **TOPIC_MODEL,
    "clustering_method": "fast_hdbscan",
    "cluster_params": {},
    "toponymy_cluster_params": {},
    "toponymy_evoc_cluster_params": {},
    "toponymy_layer_index": 1,
}
session.set_section("topic_model", TOPIC_MODEL)


Snapshot of configuration saved to config_used.yaml (9 parameters tracked).


### 5.5 Backend & LLM Configuration

**Backend matrix:**

| Backend | Clustering Input | LLM Providers | Notes |
|---------|-----------------|---------------|-------|
| `bertopic` | 5D reduced vectors | `local`, `gguf`, `huggingface_api`, `openrouter` | Standard BERTopic + outlier reduction |
| `toponymy` | 5D reduced vectors | `local`, `gguf`, `openrouter` | Hierarchical layers, richer labeling |
| `toponymy_evoc` | Raw embeddings | `local`, `gguf`, `openrouter` | EVoC clusterer, no 5D needed |

**Representation pipeline** (BERTopic):

| Model | Role | Configurable |
|-------|------|--------------|
| `POS` | Part-of-speech filtering (keep nouns, adjectives) | `pos_spacy_model` |
| `KeyBERT` | Semantic keyword re-ranking | Requires local embedding model |
| `MMR` | Maximal Marginal Relevance (diversity) | `mmr_diversity` (0–1) |
| LLM | Final topic label generation | Provider, model, prompt |

Default pipeline: **POS → KeyBERT → MMR → LLM** (sequential). Parallel models
stored separately in `topic_aspects_` for comparison.

`MIN_DF` guidance: `max(1, min(5, n_docs // 100))`. Small samples need `min_df=1`;
large corpora benefit from higher values to suppress noise terms.


### 5.5a LLM Prompt for Topic Labels

Choose a named prompt via `TOPIC_MODEL["llm_prompt_name"]` or provide a full override via
`TOPIC_MODEL["llm_prompt"]`.

Available prompt names:

- `physics`: gravitational physics, astrophysics, cosmology
- `generic`: domain-agnostic scientific labeling


In [22]:
TOPIC_MODEL = {
    **TOPIC_MODEL,
    "llm_prompt_name": "physics",
    "llm_prompt": None,
}
session.set_section("topic_model", TOPIC_MODEL)


Snapshot of configuration saved to config_used.yaml (9 parameters tracked).


In [23]:
TOPIC_MODEL = {
    **TOPIC_MODEL,
    "backend": "bertopic",
    "llm_provider": "openrouter",
    "llm_model": "google/gemini-3-flash-preview",
    "llm_api_key": None,
    "bertopic_label_max_tokens": 128,
    "toponymy_local_label_max_tokens": 256,
    "pipeline_models": ["POS", "KeyBERT", "MMR"],
    "parallel_models": ["MMR", "POS", "KeyBERT"],
    "toponymy_embedding_model": None,
    "toponymy_max_workers": 10,
    "min_df": None,
    "outlier_threshold": 0.5,
}
session.set_section("topic_model", TOPIC_MODEL)


Snapshot of configuration saved to config_used.yaml (9 parameters tracked).


### 5.5b Fit Topic Model

Run the selected backend (BERTopic or Toponymy) on reduced vectors.
This is the most compute/cost-intensive step of the pipeline.

In [24]:
session.run_stage("topic_fit")


=== Stage: topic_fit ===
Topic defaults | docs=382 | min_df=3 | min_cluster_size=15 | base_min_cluster_size=55
PyTorch version 2.6.0+cpu available.
Preparing BERTopic components (pipeline=['POS', 'KeyBERT', 'MMR'], parallel=['MMR', 'POS', 'KeyBERT']) ...
  Initializing POS keyword extraction model: en_core_web_md
Use pytorch device_name: cpu
Load pretrained SentenceTransformer: sentence-transformers/all-MiniLM-L6-v2


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

Fitting BERTopic (LLM: openrouter/google/gemini-3-flash-preview) ...


BERTopic fit:   0%|          | 0/1 [00:00<?, ?it/s]

2026-03-09 20:28:37,743 - BERTopic - Dimensionality - Fitting the dimensionality reduction algorithm
2026-03-09 20:28:37,745 - BERTopic - Dimensionality - Completed ✓
2026-03-09 20:28:37,748 - BERTopic - Cluster - Start clustering the reduced embeddings
2026-03-09 20:28:44,401 - BERTopic - Cluster - Completed ✓
2026-03-09 20:28:44,408 - BERTopic - Representation - Fine-tuning topics using representation models.
100%|██████████| 5/5 [00:08<00:00,  1.73s/it]
2026-03-09 20:29:04,903 - BERTopic - Representation - Completed ✓
  OpenRouter cost: $0.0015 (5/5 priced; direct=5, fetched=0, mode=hybrid)
  Outliers: 93 → 27
  Refreshing topic representations after outlier reassignment (not a full BERTopic refit).
  Topic reduction must happen before this step when using manual topic assignments.


BERTopic refresh:   0%|          | 0/1 [00:00<?, ?it/s]

100%|██████████| 5/5 [00:07<00:00,  1.46s/it]
  OpenRouter cost: $0.0014 (5/5 priced; direct=5, fetched=0, mode=hybrid)
  llm_labeling | tokens=2,665 | cost=$0.0015
  llm_labeling_post_outliers | tokens=2,625 | cost=$0.0014


In [25]:
if session.topic_info is not None:
    display(session.topic_info)


,Topic,Count,Name,Representation,MMR,POS,KeyBERT,Representative_Docs
0,-1,27,-1_General Relativistic and Covariant Vortex D...,[General Relativistic and Covariant Vortex Dyn...,"[helmholtz, general relativistic, covariant, u...","[theorem, relativistic, general, covariant, fo...","[helmholtz, general relativistic, general cova...",[The General-Relativistic and Covariant Form o...
1,0,142,0_Historical Foundations of Newtonian and Rela...,[Historical Foundations of Newtonian and Relat...,"[physics, cosmology, solar, quantum, geophysic...","[physics, cosmology, solar, quantum, physical,...","[cosmology, cosmological, physics, planckions,...",[Newton's Principia Mathematica Philosophiae a...
2,1,105,1_Unified Field Theories and General Relativis...,[Unified Field Theories and General Relativist...,"[equations, einstein, general relativity, grav...","[field, equations, general, theory, relativity...","[general theory relativity, general relativity...","[Hermitian Relativity, Chromodynamics, and Con..."
3,2,60,2_Mach's Principle and Relativistic Inertial D...,[Mach's Principle and Relativistic Inertial Dy...,"[newtonian, mach einstein, gravitation, inerti...","[gravitation, gravitational, inertia, principl...","[cosmology, cosmological, general relativity, ...",[Remarks on anisotropy of inertia in an anisot...
4,3,48,3_Gravitational Effects on Light and Mass Equi...,[Gravitational Effects on Light and Mass Equiv...,"[gravitation, theory gravitation, solar, equiv...","[light, gravitational, gravitation, gravity, t...","[general relativity, gravitational theory, the...","[Aberration, Gravity, and Mass Equivalence of ..."


### 5.5c Build Topic DataFrame

Merge topic assignments, 2D coordinates, and embeddings into the main DataFrame.

In [26]:
session.run_stage("topic_dataframe")


=== Stage: topic_dataframe ===


In [27]:
if session.topic_df is not None:
    display(session.topic_df.head(10))


,Bibcode,Author,Title,Year,Journal,Journal Abbreviation,Issue,Volume,First Page,Last Page,...,Abstract_en,full_text,tokens,embedding_2d_x,embedding_2d_y,topic_id,Name,MMR,POS,KeyBERT
0,1971grun.book.....T,"[Treder, H.J.]",Gravitationstheorie und Äquivalenzprinzip.,1971,Gravitationstheorie und Äquivalenzprinzip,grun.book,,,,,...,,Theory of Gravitation and Principle of Equival...,"[theory, gravitation, principle, equivalence]",-0.132630,1.807125,3,3_Gravitational Effects on Light and Mass Equi...,"gravitation, theory gravitation, solar, equiva...","light, gravitational, gravitation, gravity, th...","general relativity, gravitational theory, theo..."
1,1967AnP...475..194T,"[Treder, HansJürgen]",Das makroskopische Gravitationsfeld in der ein...,1967,Annalen der Physik,AnP,3,475,194,206,...,A theory of the gravitation field is proposed ...,The Macroscopic Gravitational Field in the Uni...,"[macroscopic, gravitational, field, unified, q...",-1.870032,-0.502315,1,1_Unified Field Theories and General Relativis...,"equations, einstein, general relativity, gravi...","field, equations, general, theory, relativity,...","general theory relativity, general relativity,..."
2,1983AN....304..145T,"[Treder, H.J.]",The problem of the Grand Unification Theory,1983,Astronomische Nachrichten,AN,4,304,145,151,...,The evolution and fundamental questions of phy...,The problem of the Grand Unification Theory. T...,"[problem, grand, unification, theory, evolutio...",-1.098783,-0.268287,1,1_Unified Field Theories and General Relativis...,"equations, einstein, general relativity, gravi...","field, equations, general, theory, relativity,...","general theory relativity, general relativity,..."
3,1957AnP...454..369T,"[Treder, H.]",Stromladungsdefinition und elektrische Kraft i...,1957,Annalen der Physik,AnP,6,454,369,380,...,"As Infeld showed in 1950, the strong system of...",Definition of Electric Charge and Electric For...,"[definition, electric, charge, electric, force...",-1.757049,-0.314226,1,1_Unified Field Theories and General Relativis...,"equations, einstein, general relativity, gravi...","field, equations, general, theory, relativity,...","general theory relativity, general relativity,..."
4,1972drdt.book.....T,"[Treder, H.J.]",Die Relativität der Trägheit.,1972,Die Relativität der Trägheit,drdt.book,,,,,...,,The Relativity of Inertia..,"[relativity, inertia]",0.353057,1.015750,3,3_Gravitational Effects on Light and Mass Equi...,"gravitation, theory gravitation, solar, equiva...","light, gravitational, gravitation, gravity, th...","general relativity, gravitational theory, theo..."
5,1997GReGr..29..455V,"[von Borzeszkowski, HorstHeino, Treder, HansJü...",The Weyl-Cartan Space Problem in Purely Affine...,1997,General Relativity and Gravitation,GReGr,4,29,455,466,...,"According to Poincaré, only the ``epistemologi...",The Weyl-Cartan Space Problem in Purely Affine...,"[weyl, cartan, space, problem, purely, affine,...",-2.333146,-0.680660,1,1_Unified Field Theories and General Relativis...,"equations, einstein, general relativity, gravi...","field, equations, general, theory, relativity,...","general theory relativity, general relativity,..."
6,1975AnP...487..383T,"[Treder, H.J.]",Zur unitarisierten Gravitationstheorie mit lan...,1975,Annalen der Physik,AnP,,487,383,400,...,,On the unitarized theory of gravitation with l...,"[unitarized, theory, gravitation, short, range...",-1.290026,0.704512,1,1_Unified Field Theories and General Relativis...,"equations, einstein, general relativity, gravi...","field, equations, general, theory, relativity,...","general theory relativity, general relativity,..."
7,1981AN....302..275T,"[Treder, H. J., Jackisch, G.]",On Soldners Value of Newtonian Deflection of L...,1981,Astronomische Nachrichten,AN,6,302,275,,...,,On Soldners Value of Newtonian Deflection of L...,"[soldners, value, newtonian, deflection, light]",0.374002,0.898423,3,3_Gravitational Effects on Light and Mass Equi...,"gravitation, theory gravitation, so

### 5.6 Visualize Topics

Render an interactive topic map from 2D coordinates and topic labels.
Use this view to inspect cluster semantics before curation.


In [28]:
VISUALIZATION = {
    "enabled": True,
    "title": "ADS Bibliometric Map",
    "subtitle_template": "Topics labeled with {provider}/{model}",
    "dark_mode": True,
}
session.set_section("visualization", VISUALIZATION)


Snapshot of configuration saved to config_used.yaml (9 parameters tracked).


In [29]:
session.run_stage("visualize")


=== Stage: visualize ===
Saved: topic_map.html


### 5.7 Curate Dataset

Review the cluster summary, then remove clusters that don't fit your research question.

In [30]:
CURATION = {
    "clusters_to_remove": [],
}
session.set_section("curation", CURATION)


Snapshot of configuration saved to config_used.yaml (9 parameters tracked).


In [31]:
session.run_stage("curate")


=== Stage: curate ===
Cluster summary rows: 5
Curated dataset saved: 382 records


### Curated Dataset Artifact

The curate stage writes the curated dataset to the current run directory.
Use the log cell below to confirm the handoff artifact for citation analysis.


In [32]:
logger.info(
    "Curated dataset artifact: %s",
    session.run.paths["data"] / "publications.parquet",
)


Curated dataset artifact: c:\Users\rapha\Documents\Studium\Promotionsstudium\MPIWG\2_Notebooks\ADS_Pipeline\runs\run_20260309_202232_ADS_Curation_Run\data\publications.parquet


In [33]:
if session.curated_df is not None:
    from ads_bib.curate import get_cluster_summary

    display(get_cluster_summary(session.curated_df))
    display(session.curated_df.head(10))


,topic_id,Count,Percentage,Label
1,0,142,37.2,0_Historical Foundations of Newtonian and Rela...
2,1,105,27.5,1_Unified Field Theories and General Relativis...
3,2,60,15.7,2_Mach's Principle and Relativistic Inertial D...
4,3,48,12.6,3_Gravitational Effects on Light and Mass Equi...
0,-1,27,7.1,-1_General Relativistic and Covariant Vortex D...


,Bibcode,Author,Title,Year,Journal,Journal Abbreviation,Issue,Volume,First Page,Last Page,...,Abstract_en,full_text,tokens,embedding_2d_x,embedding_2d_y,topic_id,Name,MMR,POS,KeyBERT
0,1971grun.book.....T,"[Treder, H.J.]",Gravitationstheorie und Äquivalenzprinzip.,1971,Gravitationstheorie und Äquivalenzprinzip,grun.book,,,,,...,,Theory of Gravitation and Principle of Equival...,"[theory, gravitation, principle, equivalence]",-0.132630,1.807125,3,3_Gravitational Effects on Light and Mass Equi...,"gravitation, theory gravitation, solar, equiva...","light, gravitational, gravitation, gravity, th...","general relativity, gravitational theory, theo..."
1,1967AnP...475..194T,"[Treder, HansJürgen]",Das makroskopische Gravitationsfeld in der ein...,1967,Annalen der Physik,AnP,3,475,194,206,...,A theory of the gravitation field is proposed ...,The Macroscopic Gravitational Field in the Uni...,"[macroscopic, gravitational, field, unified, q...",-1.870032,-0.502315,1,1_Unified Field Theories and General Relativis...,"equations, einstein, general relativity, gravi...","field, equations, general, theory, relativity,...","general theory relativity, general relativity,..."
2,1983AN....304..145T,"[Treder, H.J.]",The problem of the Grand Unification Theory,1983,Astronomische Nachrichten,AN,4,304,145,151,...,The evolution and fundamental questions of phy...,The problem of the Grand Unification Theory. T...,"[problem, grand, unification, theory, evolutio...",-1.098783,-0.268287,1,1_Unified Field Theories and General Relativis...,"equations, einstein, general relativity, gravi...","field, equations, general, theory, relativity,...","general theory relativity, general relativity,..."
3,1957AnP...454..369T,"[Treder, H.]",Stromladungsdefinition und elektrische Kraft i...,1957,Annalen der Physik,AnP,6,454,369,380,...,"As Infeld showed in 1950, the strong system of...",Definition of Electric Charge and Electric For...,"[definition, electric, charge, electric, force...",-1.757049,-0.314226,1,1_Unified Field Theories and General Relativis...,"equations, einstein, general relativity, gravi...","field, equations, general, theory, relativity,...","general theory relativity, general relativity,..."
4,1972drdt.book.....T,"[Treder, H.J.]",Die Relativität der Trägheit.,1972,Die Relativität der Trägheit,drdt.book,,,,,...,,The Relativity of Inertia..,"[relativity, inertia]",0.353057,1.015750,3,3_Gravitational Effects on Light and Mass Equi...,"gravitation, theory gravitation, solar, equiva...","light, gravitational, gravitation, gravity, th...","general relativity, gravitational theory, theo..."
5,1997GReGr..29..455V,"[von Borzeszkowski, HorstHeino, Treder, HansJü...",The Weyl-Cartan Space Problem in Purely Affine...,1997,General Relativity and Gravitation,GReGr,4,29,455,466,...,"According to Poincaré, only the ``epistemologi...",The Weyl-Cartan Space Problem in Purely Affine...,"[weyl, cartan, space, problem, purely, affine,...",-2.333146,-0.680660,1,1_Unified Field Theories and General Relativis...,"equations, einstein, general relativity, gravi...","field, equations, general, theory, relativity,...","general theory relativity, general relativity,..."
6,1975AnP...487..383T,"[Treder, H.J.]",Zur unitarisierten Gravitationstheorie mit lan...,1975,Annalen der Physik,AnP,,487,383,400,...,,On the unitarized theory of gravitation with l...,"[unitarized, theory, gravitation, short, range...",-1.290026,0.704512,1,1_Unified Field Theories and General Relativis...,"equations, einstein, general relativity, gravi...","field, equations, general, theory, relativity,...","general theory relativity, general relativity,..."
7,1981AN....302..275T,"[Treder, H. J., Jackisch, G.]",On Soldners Value of Newtonian Deflection of L...,1981,Astronomische Nachrichten,AN,6,302,275,,...,,On Soldners Value of Newtonian Deflection of L...,"[soldners, value, newtonian, deflection, light]",0.374002,0.898423,3,3_Gravitational Effects on Light and Mass Equi...,"gravitation, theory gravitation, so

---
# Phase 6: Citation Networks

Compute citation networks from the curated dataset and export
to GEXF (Gephi), Graphology JSON (Sigma.js), CSV, and/or Web of Science format.

### 6.1 Citation Configuration

Set network metrics, thresholds, filters, and export formats.
These parameters define which citation structures are constructed.

In [34]:
CITATIONS = {
    "metrics": [
        "direct",
        "co_citation",
        "bibliographic_coupling",
        "author_co_citation",
    ],
    "min_counts": {
        "direct": 5,
        "co_citation": 20,
        "bibliographic_coupling": 20,
        "author_co_citation": 10,
    },
    "authors_filter": None,
    "output_format": "gexf",
}
session.set_section("citations", CITATIONS)


Config changed; invalidated in-memory state from stage 'citations'.
Snapshot of configuration saved to config_used.yaml (9 parameters tracked).


### 6.2 Build Node Table & Compute Networks

Build node metadata and compute selected citation networks.
Outputs are exported for Gephi/Sigma/CSV workflows.


In [35]:
session.run_stage("citations")


=== Stage: citations ===
All nodes: 769


Direct citation:            0/2 [00:00]

Direct citations:   0%|          | 0/382 [00:00<?, ?it/s]

GEXF nodes:   0%|          | 0/98 [00:00<?, ?it/s]

GEXF edges:   0%|          | 0/120 [00:00<?, ?it/s]

...

Author co-citation:            0/2 [00:00]

Author co-citation detail:   0%|          | 0/126 [00:00<?, ?it/s]

GEXF nodes:   0%|          | 0/5 [00:00<?, ?it/s]

GEXF edges:   0%|          | 0/59 [00:00<?, ?it/s]

  Author co-citation — 5 nodes, 59 edges, no filter, 2.9 KB
  WOS format: download_wos_export.txt


### 6.3 Export Web of Science Format

Export the curated corpus in WOS text format.
This supports downstream tooling such as CiteSpace/VOSviewer.


In [36]:
suffix = "_filtered" if session.config.citations.authors_filter else ""
logger.info(
    "WOS export artifact: %s",
    session.run.paths["data"] / f"download_wos_export{suffix}.txt",
)


WOS export artifact: c:\Users\rapha\Documents\Studium\Promotionsstudium\MPIWG\2_Notebooks\ADS_Pipeline\runs\run_20260309_202232_ADS_Curation_Run\data\download_wos_export.txt


---
# Summary

Final dataset statistics and output file listing.

In [37]:
logger.info("Run artifacts: %s", session.run.paths["root"])


Run artifacts: c:\Users\rapha\Documents\Studium\Promotionsstudium\MPIWG\2_Notebooks\ADS_Pipeline\runs\run_20260309_202232_ADS_Curation_Run


In [38]:
session.save_summary()


PIPELINE COMPLETE
Publications:     382
References:       493
Curated dataset:  382
Topics found:     5

Output files:
  config_used.yaml (0.0 MB)
  data\author_co_citation.gexf (0.0 MB)
  data\direct.gexf (0.4 MB)
  data\download_wos_export.txt (0.3 MB)
  data\publications.parquet (0.3 MB)
  data\publications_disambiguated.parquet (0.3 MB)
  data\publications_tokenized.json (0.7 MB)
  data\publications_translated.json (0.5 MB)
  data\references.parquet (0.3 MB)
  data\references_disambiguated.parquet (0.3 MB)
  data\references_translated.json (0.6 MB)
  plots\topic_map.html (0.2 MB)

CostTracker Summary (compact)
  translation | google/gemini-3-flash-preview | tokens(total=51,183, prompt=36,621, completion=14,562) | calls=2 | cost=$0.0620
  embeddings | google/gemini-embedding-001 | tokens(total=31,197, prompt=31,197, completion=0) | calls=1 | cost=n/a
  llm_labeling | google/gemini-3-flash-preview | tokens(total=2,665, prompt=2,617, completion=48) | calls=1 | cost=$0.0015
  llm_label